# A Practical Guide to Quantitative Finance Interviews — Chapter 4 Probability Theory

**Xinfeng Zhou ("the Green Book")** — worked solutions in code.

Third notebook in the set, alongside the Chapter 2 brain teasers
([`quant_finance_interviews.ipynb`](quant_finance_interviews.ipynb)) and the Chapter 3 calculus
notes ([`quant_finance_calculus.ipynb`](quant_finance_calculus.ipynb)). Probability problems
usually *do* carry a parameter to generalise, so — like Chapter 2 — each gets a short
explanation plus **one general-purpose function** (solving for all `n`), with a Monte-Carlo
check where a closed form is subtle.

Covered so far:

| § | Topic |
|---|-------|
| **4.1** | Basic Probability Definitions & Set Operations |

## 4.1 Basic Probability Definitions and Set Operations

**The vocabulary.** A random experiment has a **sample space** $\Omega$ (all possible
outcomes); an **event** $A\subseteq\Omega$ is a set of outcomes. A probability measure $P$
assigns each event a number obeying **Kolmogorov's axioms**:
1. $P(A)\ge0$;
2. $P(\Omega)=1$;
3. for mutually exclusive events, $P(A_1\cup A_2\cup\cdots)=\sum_i P(A_i)$.

**Set operations on events** (the algebra of "and / or / not"):
- **Union** $A\cup B$ — $A$ *or* $B$ occurs;
- **Intersection** $A\cap B$ — $A$ *and* $B$ occur;
- **Complement** $A^{c}$ — $A$ does *not* occur, with $P(A^{c})=1-P(A)$;
- **De Morgan:** $(A\cup B)^{c}=A^{c}\cap B^{c}$ and $(A\cap B)^{c}=A^{c}\cup B^{c}$.

**Inclusion–exclusion** repairs the double-counting in a union:
$$P(A\cup B)=P(A)+P(B)-P(A\cap B),$$
$$P(A\cup B\cup C)=P(A)+P(B)+P(C)-P(A\cap B)-P(A\cap C)-P(B\cap C)+P(A\cap B\cap C).$$

**Conditional probability & independence.**
$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}\ \ (P(B)>0),\qquad
A,B\ \text{independent}\iff P(A\cap B)=P(A)\,P(B).$$
Two more workhorses follow: the **law of total probability**
$P(A)=\sum_i P(A\mid B_i)P(B_i)$ over a partition $\{B_i\}$, and **Bayes' rule**
$P(B_i\mid A)=\dfrac{P(A\mid B_i)P(B_i)}{\sum_j P(A\mid B_j)P(B_j)}$.

**Two recurring tricks**, used all over this section:
- **Symmetry** — if two events are interchangeable by relabelling, they carry equal
  probability (so when they also split the remaining "no-tie" cases, each is half of it).
- **Complementary counting** — often $P(A)=1-P(A^{c})$ is far easier than attacking $P(A)$ head-on.

The four problems below are pure applications of these basics.

### 4.1.1 Coin toss game

**Problem.** Gambler $A$ flips $n+1$ fair coins and gambler $B$ flips $n$. What is the
probability that $A$ gets **strictly more heads** than $B$?

**Logic (symmetry — essentially no computation).** Let $H_A,H_B$ be the head counts and
$T_A,T_B$ the tail counts, so $H_A+T_A=n+1$ and $H_B+T_B=n$. Consider
$$X:\ H_A>H_B\ \text{("A has more heads")},\qquad Y:\ T_A>T_B\ \text{("A has more tails")}.$$
- **They are mutually exclusive.** If both held, then $(H_A-H_B)+(T_A-T_B)=(n+1)-n=1$ with
  each difference $\ge1$ — impossible (the two would sum to $\ge2$).
- **One of them always holds.** If neither did, then $H_A\le H_B$ *and* $T_A\le T_B$, forcing
  $n+1=H_A+T_A\le H_B+T_B=n$ — impossible.

So $X$ and $Y$ **partition** the sample space: $P(X)+P(Y)=1$. And since the coins are fair,
swapping heads$\leftrightarrow$tails is a symmetry carrying $X$ to $Y$, so $P(X)=P(Y)$. Hence
$$\boxed{\,P(H_A>H_B)=\tfrac12\,}\qquad\text{for every }n.$$
The function below confirms it by summing the exact binomial probabilities.

In [1]:
from math import comb
from fractions import Fraction

def coin_toss_A_more_heads(n):
    """P(A's heads > B's heads) when A flips n+1 fair coins and B flips n, computed
    exactly from the binomial pmfs. The symmetry argument proves it is 1/2 for every n."""
    total = sum(comb(n + 1, a) * comb(n, b)
                for a in range(n + 2) for b in range(n + 1) if a > b)
    return Fraction(total, 2 ** (2 * n + 1))     # divide by 2^{(n+1)+n}

for n in [0, 1, 2, 5, 10, 25]:
    print(f"n={n:>2}: P(A has more heads) = {coin_toss_A_more_heads(n)}")

n= 0: P(A has more heads) = 1/2
n= 1: P(A has more heads) = 1/2
n= 2: P(A has more heads) = 1/2
n= 5: P(A has more heads) = 1/2
n=10: P(A has more heads) = 1/2
n=25: P(A has more heads) = 1/2


### 4.1.2 Card game

**Problem.** A $52$-card deck has $13$ values ($2,3,\dots,10,J,Q,K,A$), four cards each. You
draw one card and the dealer draws another **without replacement**. You win only if your value
is **strictly higher**; on a tie or a lower value the house wins. What is your winning
probability?

**Logic (symmetry + a single tie term).** Your card and the dealer's are exchangeable, so by
symmetry $P(\text{you}>\text{dealer})=P(\text{dealer}>\text{you})$. Together with a **tie**
these exhaust the outcomes, so
$$P(\text{win})=\frac{1-P(\text{tie})}{2}.$$
Only the tie needs counting: after you draw a card, $51$ remain and $3$ of them share your
value, so $P(\text{tie})=\dfrac{3}{51}=\dfrac1{17}$. Therefore
$$P(\text{win})=\frac{1-\frac1{17}}{2}=\frac{16/17}{2}=\boxed{\dfrac{8}{17}}\approx0.4706,$$
just below a coin flip — the house keeps its edge. In general, for $v$ values with $k$ copies
each ($N=vk$ cards), $P(\text{tie})=\dfrac{k-1}{N-1}$ and
$P(\text{win})=\dfrac12\!\left(1-\dfrac{k-1}{N-1}\right)$.

In [2]:
from fractions import Fraction

def card_win_prob(values=13, copies=4):
    """P(your card's value > dealer's) drawing two cards without replacement from a deck of
    `values` distinct values, `copies` each. P(tie)=(copies-1)/(N-1) and, by symmetry,
    P(win)=(1-P(tie))/2. Returns (P(win), P(tie))."""
    N = values * copies
    p_tie = Fraction(copies - 1, N - 1)
    return (1 - p_tie) / 2, p_tie

w, t = card_win_prob(13, 4)
print(f"standard deck (13 values x 4): P(win) = {w} = {float(w):.4f},  P(tie) = {t}")
for v, k in [(13, 1), (5, 10), (13, 4)]:
    w, t = card_win_prob(v, k)
    print(f"{v:>2} values x {k:>2}: P(win) = {w} = {float(w):.4f}")

standard deck (13 values x 4): P(win) = 8/17 = 0.4706,  P(tie) = 1/17
13 values x  1: P(win) = 1/2 = 0.5000
 5 values x 10: P(win) = 20/49 = 0.4082
13 values x  4: P(win) = 8/17 = 0.4706


### 4.1.3 Drunk passenger

**Problem.** $100$ passengers board in order; passenger $n$ owns seat $n$. The **first**
passenger is drunk and takes a **uniformly random** seat. Everyone after sits in their own seat
if it is free, otherwise in a uniformly random free seat. What is the probability **you**
(passenger $100$) end up in your own seat?

**Logic (a clean symmetry).** Follow the "displaced" passenger — initially the drunk one. Each
time a displaced passenger picks at random, the process ends the moment someone sits in **seat
$1$** (then every later passenger, you included, finds their own seat free → you **win**) or in
**seat $100$** (then you are ultimately bumped → you **lose**). Any random pick that hits some
*other* seat merely hands the "displaced" role to a new passenger and continues. So the outcome
is a race between seat $1$ and seat $100$, and at each random choice those two seats are
**equally likely** to be taken — so by symmetry the process ends on each with probability
$\tfrac12$:
$$\boxed{\,P(\text{you get seat }100)=\tfrac12\,}\qquad(n\ge2).$$
Strikingly this is $\tfrac12$ for **any** number of passengers $\ge2$ — the $100$ is a red
herring. Exact value and a Monte-Carlo check below.

In [3]:
import random
from fractions import Fraction

def drunk_passenger_exact(n):
    """P(passenger n gets their own seat) in the drunk-first-passenger problem:
    exactly 1/2 for n >= 2 (and 1 for n = 1)."""
    return Fraction(1) if n <= 1 else Fraction(1, 2)

def drunk_passenger_sim(n, trials=20000, seed=0):
    """Monte-Carlo estimate of the same probability, boarding the plane `trials` times."""
    rng = random.Random(seed)
    wins = 0
    for _ in range(trials):
        taken = bytearray(n + 1)                       # seats 1..n (index 0 unused)
        taken[rng.randint(1, n)] = 1                    # drunk passenger 1
        for i in range(2, n):                           # sober passengers 2..n-1
            if not taken[i]:
                taken[i] = 1                            # own seat free -> take it
            else:
                free = [s for s in range(1, n + 1) if not taken[s]]
                taken[rng.choice(free)] = 1             # else a random free seat
        wins += (taken[n] == 0)                          # you win iff seat n is still free
    return wins / trials

for n in [2, 5, 100]:
    print(f"n={n:>3}: exact = {drunk_passenger_exact(n)}   simulated = {drunk_passenger_sim(n):.3f}")

n=  2: exact = 1/2   simulated = 0.503
n=  5: exact = 1/2   simulated = 0.495


n=100: exact = 1/2   simulated = 0.503


### 4.1.4 N points on a circle

**Problem.** $N$ points are dropped uniformly at random on a circle's circumference. What is the
probability they **all lie within some semicircle**?

**Logic (fix a point, then sum disjoint cases).** For each point $i$, let $E_i$ be the event
"starting at point $i$ and sweeping **clockwise**, the next half-circle contains all the other
$N-1$ points." Each other point independently falls in that fixed semicircle with probability
$\tfrac12$, so
$$P(E_i)=\left(\tfrac12\right)^{N-1}.$$
The events $E_1,\dots,E_N$ are **mutually exclusive** — at most one point can be the clockwise
"first" point of an all-containing semicircle. Since "all in some semicircle" is exactly "some
$E_i$ occurs,"
$$P(\text{all within a semicircle})=\sum_{i=1}^{N}P(E_i)=\boxed{\dfrac{N}{2^{\,N-1}}}.$$
Sanity: $N=2\Rightarrow1$ (two points always fit), $N=3\Rightarrow\tfrac34$,
$N=4\Rightarrow\tfrac12$. The function returns the exact value; the simulation checks it via the
equivalent test *"the largest gap between adjacent points is at least half the circle."*

In [4]:
import random
from fractions import Fraction

def semicircle_prob(N):
    """P(all N uniformly-random points on a circle lie within some semicircle) = N / 2^{N-1}."""
    return Fraction(N, 2 ** (N - 1))

def semicircle_sim(N, trials=200000, seed=0):
    """Monte-Carlo check: all points fit in a semicircle iff some gap between adjacent points
    (around the circle) is at least half the circumference."""
    rng = random.Random(seed)
    hits = 0
    for _ in range(trials):
        pts = sorted(rng.random() for _ in range(N))        # positions as fractions of the circle
        gaps = [pts[i + 1] - pts[i] for i in range(N - 1)] + [1 - pts[-1] + pts[0]]
        hits += (max(gaps) >= 0.5)
    return hits / trials

for N in [2, 3, 4, 5, 6]:
    exact = semicircle_prob(N)
    print(f"N={N}: exact = {exact} = {float(exact):.4f}   simulated = {semicircle_sim(N):.4f}")

N=2: exact = 1 = 1.0000   simulated = 1.0000


N=3: exact = 3/4 = 0.7500   simulated = 0.7506


N=4: exact = 1/2 = 0.5000   simulated = 0.5011


N=5: exact = 5/16 = 0.3125   simulated = 0.3131


N=6: exact = 3/16 = 0.1875   simulated = 0.1882


---
*More of Chapter 4 as I keep reading: combinatorics, conditional probability, and the standard
distributions.*